# Test Finetuning Methods in VQAModel

This notebook tests:
1. Existing functionality still works (encode, generate, score_choices)
2. New `get_loss()` method for batch training
3. New `prepare_training_batch()` method for editor compatibility


In [15]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()
import os, sys
os.environ["TORCH_NVML_DISABLED"] = "1"
os.chdir("/scratch/jq2uw/MME/instruct_vlm_edit")

repo_root = "/scratch/jq2uw/MME/instruct_vlm_edit"
if repo_root not in sys.path:
    sys.path.append(repo_root)

from revlm.config_utils import *
from revlm.dataset import *
from revlm.models import *
from revlm.editors import *
import argparse

MODELNAME = "blip"


In [16]:
# Setup config and model
cfg_path = os.path.join(repo_root, "revlm", "config", "config.yaml")

args = argparse.Namespace(
    config=cfg_path,
    editor="ft",
    inner_params=[],  # Will auto-select if empty
    dataset_name="aokvqa",
    model_name=MODELNAME,
    batch_size=2,  # Start with smaller batch to avoid OOM
    seed=42,
    task="mc",
)

config = configure_args(args, config_path=cfg_path)
config.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Config device: {config.device}")
config


Task evaluation metrics will be saved to results/te/ft/instructblip-vicuna-7b/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/instructblip-vicuna-7b/aokvqa
Predictions will be saved to results/pred/instructblip-vicuna-7b/aokvqa
Unified filename to save: mc_all.json
Config device: cuda:0


namespace(batch_size=2,
          n_iter=100,
          max_n_edits=5000,
          seed=42,
          device=device(type='cuda', index=0),
          ckpt_dir=None,
          dropout=None,
          task_dir='results/te/ft/instructblip-vicuna-7b/aokvqa',
          edit_dir='results/ee/ft/instructblip-vicuna-7b/aokvqa',
          pred_dir='results/pred/instructblip-vicuna-7b/aokvqa',
          fname='mc_all.json',
          model=namespace(name='Salesforce/instructblip-vicuna-7b',
                          class_name='VQAModel',
                          pt=None,
                          inner_params=['language_projection.weight'],
                          processor_class=None,
                          tokenizer_class=None,
                          temperature=0.0),
          editor=namespace(_name='ft', edit_lr='1e-4'),
          experiment=namespace(task='mc',
                               dataset_name='aokvqa',
                               pred_by='label_maxprob',
            

In [17]:
# Load model
vlm = VQAModel(config)
print(f"Model loaded: {type(vlm.model).__name__}")
print(f"Model device: {next(vlm.model.parameters()).device}")


The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Model loaded: InstructBlipForConditionalGeneration
Model device: cuda:0


## Test 1: Existing Functionality (Should still work)


In [18]:
# Test encode() - should work as before
from PIL import Image
import glob

# Load a test image from local data/images directory
try:
    # Try to find any image in data/images
    image_paths = []
    for ext in ['*.jpg', '*.jpeg', '*.png']:
        image_paths.extend(glob.glob(os.path.join(repo_root, 'data', 'images', '**', ext), recursive=True))
        if image_paths:
            break
    
    if image_paths:
        test_image_path = image_paths[0]
        test_image = Image.open(test_image_path).convert('RGB')
        print(f"Loaded test image: {test_image_path}")
    else:
        raise FileNotFoundError("No images found in data/images directory")
    
    test_prompt = "What is in this image?"
    inputs = vlm.encode([test_image], [test_prompt], tokenize=False)
    print(f"✓ encode() works. Input keys: {list(inputs.keys())}")
except Exception as e:
    print(f"✗ encode() failed: {e}")
    import traceback
    traceback.print_exc()


Loaded test image: /scratch/jq2uw/MME/instruct_vlm_edit/data/images/fvqa/COCO_val2014_000000010363.jpg
✓ encode() works. Input keys: ['qformer_input_ids', 'qformer_attention_mask', 'input_ids', 'attention_mask', 'pixel_values']


In [19]:
# Test generate() - should work as before
try:
    answers = vlm.generate([test_image], [test_prompt], max_new_tokens=50)
    print(f"✓ generate() works. Answer: {answers[0]}")
except Exception as e:
    print(f"✗ generate() failed: {e}")


✓ generate() works. Answer: In this image, there is a cat sitting on the hood of a black car in a garage. The cat is standing on the car, looking around and possibly exploring its surroundings. There are also various items in the garage


In [20]:
# Test score_choices_single() - should work as before
try:
    choices = ["cat", "dog", "bird", "car"]
    scores = vlm.score_choices_single(test_image, test_prompt, choices)
    print(f"✓ score_choices_single() works.")
    print(f"  Best choice: {max(scores, key=lambda k: scores[k]['prob'])} (prob: {max(scores.values(), key=lambda v: v['prob'])['prob']:.4f})")
except Exception as e:
    print(f"✗ score_choices_single() failed: {e}")


✓ score_choices_single() works.
  Best choice: cat (prob: 0.9367)


## Test 2: New get_loss() Method


In [21]:
# Load a small batch from dataset
train_dataset = VQADataset(config)
train_dataset.set_dataloader()

# Get one batch
batch = next(iter(train_dataset.loader))
print(f"Batch keys: {list(batch.keys())}")
print(f"Batch size: {len(batch['images'])}")
print(f"Sample gold label: {batch['golds'][0]['label']}")


Batch keys: ['images', 'prompts', 'golds', 'idxs']
Batch size: 2
Sample gold label: air purifier


In [22]:
# Test get_loss() - should return a loss tensor with gradients
try:
    # Clear GPU cache first
    torch.cuda.empty_cache()
    vlm.model.zero_grad()
    vlm.model.eval()
    
    # Use smaller batch to avoid OOM
    train_dataset.set_dataloader(
        with_rationale=False,
        shuffle_choices=False
    )
    small_batch = next(iter(train_dataset.loader))
    
    # Enable training mode for gradients
    vlm.model.train()
    
    loss = vlm.get_loss(small_batch)
    print(f"✓ get_loss() works.")
    print(f"  Loss value: {loss.item():.4f}")
    print(f"  Loss requires_grad: {loss.requires_grad}")
    print(f"  Loss dtype: {loss.dtype}")
    
    # Test backward pass
    loss.backward()
    print(f"  ✓ Backward pass successful")
    
    # Check if any parameters have gradients
    has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in vlm.model.parameters() if p.requires_grad)
    print(f"  Has gradients: {has_grad}")
    
    # Clean up
    vlm.model.zero_grad()
    vlm.model.eval()
    del small_batch, loss
    torch.cuda.empty_cache()
    
except Exception as e:
    import traceback
    print(f"✗ get_loss() failed: {e}")
    traceback.print_exc()
    vlm.model.eval()
    vlm.model.zero_grad()
    torch.cuda.empty_cache()


✓ get_loss() works.
  Loss value: 5.3154
  Loss requires_grad: True
  Loss dtype: torch.float32
  ✓ Backward pass successful
  Has gradients: True


## Test 3: New prepare_training_batch() Method


In [23]:
# Test prepare_training_batch() - should return full model inputs dict
try:
    model_inputs = vlm.prepare_training_batch(batch)
    print(f"✓ prepare_training_batch() works.")
    print(f"  Input keys: {list(model_inputs.keys())}")
    print(f"  Has 'labels': {'labels' in model_inputs}")
    
    if 'labels' in model_inputs:
        labels = model_inputs['labels']
        print(f"  Labels shape: {labels.shape}")
        print(f"  Labels dtype: {labels.dtype}")
        # Check label masking (should have -100 for prompt positions)
        n_masked = (labels == -100).sum().item()
        n_unmasked = (labels != -100).sum().item()
        print(f"  Masked tokens (-100): {n_masked}, Unmasked tokens: {n_unmasked}")
    
    # Check for image inputs
    image_keys = [k for k in model_inputs.keys() if 'pixel' in k.lower()]
    if image_keys:
        print(f"  Image input keys: {image_keys}")
    
except Exception as e:
    import traceback
    print(f"✗ prepare_training_batch() failed: {e}")
    traceback.print_exc()


✓ prepare_training_batch() works.
  Input keys: ['qformer_input_ids', 'qformer_attention_mask', 'input_ids', 'attention_mask', 'pixel_values', 'labels']
  Has 'labels': True
  Labels shape: torch.Size([2, 71])
  Labels dtype: torch.int64
  Masked tokens (-100): 136, Unmasked tokens: 6
  Image input keys: ['pixel_values']


In [24]:
# Test that model(**model_inputs) works and produces loss
try:
    # Clear GPU cache and gradients first
    torch.cuda.empty_cache()
    vlm.model.zero_grad()
    vlm.model.eval()
    
    # Use a smaller batch to avoid OOM - create fresh batch with batch_size=1
    train_dataset.set_dataloader(
        with_rationale=False,
        shuffle_choices=False
    )
    small_batch = next(iter(train_dataset.loader))
    model_inputs_small = vlm.prepare_training_batch(small_batch)
    
    # Now test with smaller batch
    vlm.model.train()
    
    outputs = vlm.model(**model_inputs_small)
    print(f"✓ model(**model_inputs) works.")
    print(f"  Has 'loss' attribute: {hasattr(outputs, 'loss')}")
    
    if hasattr(outputs, 'loss'):
        loss = outputs.loss
        print(f"  Loss value: {loss.item():.4f}")
        print(f"  Loss requires_grad: {loss.requires_grad}")
        
        # Test backward
        loss.backward()
        print(f"  ✓ Backward pass successful")
        vlm.model.zero_grad()
    
    # Clean up
    vlm.model.eval()
    del model_inputs_small, outputs, small_batch
    torch.cuda.empty_cache()
    
except Exception as e:
    import traceback
    print(f"✗ model(**model_inputs) failed: {e}")
    traceback.print_exc()
    # Ensure cleanup even on error
    vlm.model.eval()
    vlm.model.zero_grad()
    torch.cuda.empty_cache()


✓ model(**model_inputs) works.
  Has 'loss' attribute: True
  Loss value: 11.3333
  Loss requires_grad: True
  ✓ Backward pass successful


## Test 4: Compare get_loss() vs prepare_training_batch() + model forward


In [25]:
# They should produce the same loss value
try:
    # Clear GPU cache and use smaller batch
    torch.cuda.empty_cache()
    vlm.model.zero_grad()
    
    train_dataset.set_dataloader(
        with_rationale=False,
        shuffle_choices=False
    )
    small_batch = next(iter(train_dataset.loader))
    
    vlm.model.eval()  # For consistent comparison
    
    # Method 1: Direct get_loss()
    loss1 = vlm.get_loss(small_batch)
    
    # Method 2: prepare_training_batch() + model forward
    model_inputs = vlm.prepare_training_batch(small_batch)
    outputs = vlm.model(**model_inputs)
    loss2 = outputs.loss
    
    print(f"Loss from get_loss(): {loss1.item():.4f}")
    print(f"Loss from prepare_training_batch() + model: {loss2.item():.4f}")
    print(f"Difference: {abs(loss1.item() - loss2.item()):.6f}")
    
    if abs(loss1.item() - loss2.item()) < 1e-5:
        print("✓ Losses match!")
    else:
        print("⚠ Losses differ slightly (may be due to model state differences)")
    
    # Clean up
    del small_batch, model_inputs, outputs, loss1, loss2
    torch.cuda.empty_cache()
        
except Exception as e:
    import traceback
    print(f"✗ Comparison failed: {e}")
    traceback.print_exc()
    vlm.model.eval()
    vlm.model.zero_grad()
    torch.cuda.empty_cache()


Loss from get_loss(): 4.3314
Loss from prepare_training_batch() + model: 4.3314
Difference: 0.000000
✓ Losses match!


## Test 5: Test with Rationale


In [26]:
# Test with rationale=True
try:
    torch.cuda.empty_cache()
    vlm.model.zero_grad()
    
    train_dataset.set_dataloader(
        with_rationale=True,  # Enable rationale
        shuffle_choices=False,
    )

    batch_with_rationale = next(iter(train_dataset.loader))
    print(f"Sample prompt with rationale: {batch_with_rationale['prompts'][0][:200]}...")

    loss = vlm.get_loss(batch_with_rationale)
    print(f"✓ get_loss() works with rationale. Loss: {loss.item():.4f}")
    
    model_inputs = vlm.prepare_training_batch(batch_with_rationale)
    print(f"✓ prepare_training_batch() works with rationale.")
    print(f"  Input IDs shape: {model_inputs.get('input_ids', 'N/A').shape if 'input_ids' in model_inputs else 'N/A'}")
    
    # Clean up
    del batch_with_rationale, model_inputs, loss
    torch.cuda.empty_cache()
    
except Exception as e:
    import traceback
    print(f"✗ Failed with rationale: {e}")
    traceback.print_exc()
    vlm.model.eval()
    vlm.model.zero_grad()
    torch.cuda.empty_cache()


Sample prompt with rationale: Choose the correct answer from the options. What gave the cheese that consistency? Options: starch; cold; salt; heat The cheese topping of this dish shows signs of being melted over it....
✓ get_loss() works with rationale. Loss: 10.1469
✓ prepare_training_batch() works with rationale.
  Input IDs shape: torch.Size([2, 81])


## Test 6: Explore Layer Names for VLMs

Helper to find suitable layers to finetune for each model


In [27]:
# Helper function to explore layer names and suggest finetuning targets
def explore_layers(model, top_k=100):
    """Print layer names and suggest good candidates for finetuning"""
    print(f"Model: {type(model).__name__}")
    print("\n" + "="*80)
    print("All Layer Names (showing first/last few and suggested candidates):")
    print("="*80)
    
    all_names = [n for n, p in model.named_parameters()]
    
    # Show first 10 and last 10
    print("\nFirst 10 layers:")
    for name in all_names[:10]:
        print(f"  {name}")
    
    if len(all_names) > 20:
        print("\n... (hidden layers) ...")
    
    print("\nLast 10 layers:")
    for name in all_names[-10:]:
        print(f"  {name}")
    
    # Suggest good candidates (output/language head, attention, MLP)
    suggestions = []
    keywords = ['lm_head', 'embed_out', 'output', 'classifier', 'head', 
                'self_attn.q_proj', 'self_attn.v_proj', 'self_attn.k_proj',
                'mlp.c_fc', 'mlp.c_proj', 'gate_proj', 'up_proj', 'down_proj']
    
    for name in all_names:
        for kw in keywords:
            if kw in name.lower():
                suggestions.append(name)
                break
    
    print("\n" + "="*80)
    print(f"Suggested Finetuning Candidates (found {len(suggestions)}):")
    print("="*80)
    for name in suggestions[:top_k]:
        # Get param info
        param = dict(model.named_parameters())[name]
        num_params = param.numel()
        print(f"  {name}")
        print(f"    Shape: {param.shape}, Params: {num_params:,}")
    
    return suggestions[:top_k]


In [28]:
# Explore layers for current model (llava)
torch.cuda.empty_cache()
suggested_layers = explore_layers(vlm.model)
print(f"\n✓ Found {len(suggested_layers)} suggested layer candidates")
print("\nExample config entry for inner_params:")
if suggested_layers:
    print(f'  inner_params: ["{suggested_layers[0]}"]')


Model: InstructBlipForConditionalGeneration

All Layer Names (showing first/last few and suggested candidates):

First 10 layers:
  query_tokens
  vision_model.embeddings.class_embedding
  vision_model.embeddings.position_embedding
  vision_model.embeddings.patch_embedding.weight
  vision_model.embeddings.patch_embedding.bias
  vision_model.encoder.layers.0.self_attn.qkv.weight
  vision_model.encoder.layers.0.self_attn.qkv.bias
  vision_model.encoder.layers.0.self_attn.projection.weight
  vision_model.encoder.layers.0.self_attn.projection.bias
  vision_model.encoder.layers.0.layer_norm1.weight

... (hidden layers) ...

Last 10 layers:
  language_model.model.layers.31.self_attn.k_proj.weight
  language_model.model.layers.31.self_attn.v_proj.weight
  language_model.model.layers.31.self_attn.o_proj.weight
  language_model.model.layers.31.mlp.gate_proj.weight
  language_model.model.layers.31.mlp.up_proj.weight
  language_model.model.layers.31.mlp.down_proj.weight
  language_model.model.lay

## Test 7: Editor Integration with prepare_training_batch()

Test that editors work with the output from prepare_training_batch()


In [15]:
# Test Editor: ft (basic finetune)
try:
    from revlm import *
    torch.cuda.empty_cache()
    
    # Setup config for ft editor - uses YAML configs automatically
    config_ft = configure_args(argparse.Namespace(
        config=cfg_path,
        editor="ft",
        inner_params=[],  # Will use YAML preset or auto-select if empty
        dataset_name="aokvqa",
        model_name=MODELNAME,  # Short name loads model/{MODELNAME}.yaml automatically
    ), config_path=cfg_path)
    config_ft.device = config.device
    
    # Auto-select layer if not provided (from YAML or auto-select)
    if not getattr(config_ft.model, 'inner_params', []) or len(config_ft.model.inner_params) == 0:
        # Load model first to explore layers
        model_ft = VQAModel(config_ft)
        suggestions = explore_layers(model_ft.model, top_k=1)
        if suggestions:
            config_ft.model.inner_params = [suggestions[0]]
            print(f"Auto-selected layer: {config_ft.model.inner_params[0]}")
        else:
            raise ValueError("No suitable layers found")
    else:
        model_ft = VQAModel(config_ft)
    
    # Load editor
    from revlm.editors import get_editor
    editor_ft = get_editor(config_ft, model_ft)
    
    # Verify parameter exists in editor's model (auto-correct if needed)
    editor_params = dict(editor_ft.model.named_parameters())
    # if editor_ft.pnames[0] not in editor_params:
    #     print(f"⚠ Parameter '{editor_ft.pnames[0]}' not found in editor's model")
    #     layer_parts = editor_ft.pnames[0].split('.')
    #     key_parts = [p for p in layer_parts if p and not p.isdigit()]
    #     matches = [n for n in editor_params.keys() 
    #                if all(part.lower() in n.lower() for part in key_parts[-3:])]
    #     if matches:
    #         corrected_name = matches[0]
    #         config_ft.model.inner_params = [corrected_name]
    #         print(f"  Using corrected layer: {corrected_name}")
    #         editor_ft = get_editor(config_ft, model_ft, config_ft.device)
    #     else:
    #         fallback = list(editor_params.keys())[0]
    #         config_ft.model.inner_params = [fallback]
    #         print(f"  Using fallback layer: {fallback}")
    #         editor_ft = get_editor(config_ft, model_ft, config_ft.device)
    
    #         print(f"✓ Editor 'ft' loaded")
    #     print(f"  Training layer: {editor_ft.pnames}")
    #     if hasattr(config_ft.model, 'inner_params') and config_ft.model.inner_params:
    #         print(f"  Layer source: YAML preset (model/{MODELNAME}.yaml)")
    #     else:
    #         print(f"  Layer source: auto-selected")
    
    # Prepare batch using prepare_training_batch()
    train_dataset.set_dataloader(
        with_rationale=False
    )
    test_batch = next(iter(train_dataset.loader))
    tokens = model_ft.prepare_training_batch(test_batch)
    
    print(f"✓ prepare_training_batch() output keys: {list(tokens.keys())}")
    
    # Test editor.edit() with the tokens
    batch_history = []
    model_ft.model.train()  # Enable training mode
    
    print("\nRunning editor.edit()...")
    editor_ft.edit(config_ft, tokens, batch_history)
    
    print(f"✓ editor.edit() completed successfully!")
    print(f"  Loss history length: {len(editor_ft.losses)}")
    if editor_ft.losses:
        print(f"  Final loss: {editor_ft.losses[-1]:.4f}")
    
    # Clean up
    model_ft.model.eval()
    del model_ft, editor_ft, tokens, test_batch
    torch.cuda.empty_cache()
    
except Exception as e:
    import traceback
    print(f"✗ ft editor test failed: {e}")
    traceback.print_exc()
    torch.cuda.empty_cache()


Task evaluation metrics will be saved to results/te/ft/Qwen3-VL-4B-Instruct/aokvqa
Edit evaluation metrics will be saved to results/ee/ft/Qwen3-VL-4B-Instruct/aokvqa
Predictions will be saved to results/pred/Qwen3-VL-4B-Instruct/aokvqa
Unified filename to save: mc_all.json
Finetuning module model.language_model.layers.16.mlp.gate_proj
✓ prepare_training_batch() output keys: ['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw', 'labels']

Running editor.edit()...
✓ editor.edit() completed successfully!
  Loss history length: 100
  Final loss: 0.0006


### Test Editor: ft (Simple Version - No Parameter Validation)

Simpler version without parameter validation - just uses whatever layer is in config or auto-selected.


In [ ]:
# Test Editor: ft (simple version - no parameter validation)
try:
    torch.cuda.empty_cache()
    
    # Setup config for ft editor - uses YAML configs automatically
    config_ft_simple = configure_args(argparse.Namespace(
        config=cfg_path,
        editor="ft",
        inner_params=[],  # Will use YAML preset or auto-select if empty
        dataset_name="aokvqa",
        model_name=MODELNAME,  # Short name loads model/{MODELNAME}.yaml automatically
    ), config_path=cfg_path)
    config_ft_simple.device = config.device
    
    # Auto-select layer if not provided (from YAML or auto-select)
    if not getattr(config_ft_simple.model, 'inner_params', []) or len(config_ft_simple.model.inner_params) == 0:
        # Load model first to explore layers
        model_ft_simple = VQAModel(config_ft_simple)
        suggestions = explore_layers(model_ft_simple.model, top_k=1)
        if suggestions:
            config_ft_simple.model.inner_params = [suggestions[0]]
            print(f"Auto-selected layer: {config_ft_simple.model.inner_params[0]}")
        else:
            raise ValueError("No suitable layers found")
    else:
        model_ft_simple = VQAModel(config_ft_simple)
    
    print(f"Using layer: {config_ft_simple.model.inner_params[0]}")
    
    # Load editor directly - NO VALIDATION
    from revlm.editors import get_editor
    editor_ft_simple = get_editor(config_ft_simple, model_ft_simple, config_ft_simple.device)
    
    print(f"✓ Editor 'ft' loaded (simple version)")
    print(f"  Training layer: {editor_ft_simple.pnames}")
    
    # Prepare batch using prepare_training_batch()
    train_dataset.set_dataloader(
        with_rationale=False
    )
    test_batch = next(iter(train_dataset.loader))
    tokens = model_ft_simple.prepare_training_batch(test_batch)
    
    print(f"✓ prepare_training_batch() output keys: {list(tokens.keys())}")
    
    # Test editor.edit() with the tokens
    batch_history = []
    model_ft_simple.model.train()  # Enable training mode
    
    print("\nRunning editor.edit() (simple version - no validation)...")
    editor_ft_simple.edit(config_ft_simple, tokens, batch_history)
    
    print(f"✓ editor.edit() completed successfully!")
    print(f"  Loss history length: {len(editor_ft_simple.losses)}")
    if editor_ft_simple.losses:
        print(f"  Final loss: {editor_ft_simple.losses[-1]:.4f}")
    
    # Clean up
    model_ft_simple.model.eval()
    del model_ft_simple, editor_ft_simple, tokens, test_batch
    torch.cuda.empty_cache()
    
except Exception as e:
    import traceback
    print(f"✗ ft editor test (simple) failed: {e}")
    traceback.print_exc()
    torch.cuda.empty_cache()


### ft_ewc

In [ ]:
# Test Editor: ft_ewc (finetune with EWC)
try:
    import os, torch
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
    torch.cuda.empty_cache()
    
    # Setup config for ft_ewc editor - uses YAML configs automatically
    config_ewc = configure_args(argparse.Namespace(
        config=cfg_path,
        editor="ft_ewc",
        inner_params=[],  # Will use YAML preset or auto-select if empty
        dataset_name="aokvqa",
        model_name=MODELNAME,  # Short name loads model/{MODELNAME}.yaml automatically
    ), config_path=cfg_path)
    config_ewc.device = config.device
    
    # Auto-select layer if not provided (from YAML or auto-select)
    if not getattr(config_ewc.model, 'inner_params', []) or len(config_ewc.model.inner_params) == 0:
        model_ewc = get_model(config_ewc)
        suggestions = explore_layers(model_ewc.model, top_k=1)
        if suggestions:
            config_ewc.model.inner_params = [suggestions[0]]
            print(f"Auto-selected layer: {config_ewc.model.inner_params[0]}")
        else:
            raise ValueError("No suitable layers found")
    else:
        model_ewc = get_model(config_ewc)
    
    # Load editor
    from revlm.editors import get_editor
    editor_ewc = get_editor(config_ewc, model_ewc, config_ewc.device)
    
    # # Verify parameter exists in editor's model
    # editor_params = dict(editor_ewc.model.named_parameters())
    # if editor_ewc.pnames[0] not in editor_params:
    #     print(f"⚠ Parameter '{editor_ewc.pnames[0]}' not found in editor's model")
    #     layer_parts = editor_ewc.pnames[0].split('.')
    #     key_parts = [p for p in layer_parts if p and not p.isdigit()]
    #     matches = [n for n in editor_params.keys() 
    #                if all(part.lower() in n.lower() for part in key_parts[-3:])]
    #     if matches:
    #         corrected_name = matches[0]
    #         config_ewc.model.inner_params = [corrected_name]
    #         print(f"  Using corrected layer: {corrected_name}")
    #         editor_ewc = get_editor(config_ewc, model_ewc, config_ewc.device)
    #     else:
    #         fallback = list(editor_params.keys())[0]
    #         config_ewc.model.inner_params = [fallback]
    #         print(f"  Using fallback layer: {fallback}")
    #         editor_ewc = get_editor(config_ewc, model_ewc, config_ewc.device)
    
    print(f"✓ Editor 'ft_ewc' loaded")
    print(f"  Training layer: {editor_ewc.pnames}")
    print(f"  EWC lambda: {editor_ewc.ewc_lambda}")
    
    # Prepare batches for history (EWC needs batch_history)
    train_dataset.set_dataloader(
        task="mc",
        with_rationale=False,
        batch_size=1,
        shuffle=False,
    )
    
    # Create batch history (EWC uses this for Fisher matrix)
    batch_history = []
    for i, batch in enumerate(train_dataset.loader):
        if i >= 2:  # Just get 2 batches for history
            break
        tokens_hist = model_ewc.prepare_training_batch(batch)
        batch_history.append(tokens_hist)
    
    # Current batch
    test_batch = next(iter(train_dataset.loader))
    tokens = model_ewc.prepare_training_batch(test_batch)
    
    print(f"✓ Prepared {len(batch_history)} batches for history")
    print(f"✓ Current batch keys: {list(tokens.keys())}")
    
    # Test editor.edit() with EWC
    model_ewc.model.train()
    
    print("\nRunning editor.edit() with EWC...")
    editor_ewc.edit(config_ewc, tokens, batch_history)
    
    print(f"✓ editor.edit() (EWC) completed successfully!")
    print(f"  Loss history length: {len(editor_ewc.losses)}")
    if editor_ewc.losses:
        print(f"  Final loss: {editor_ewc.losses[-1]:.4f}")
    
    # Clean up
    model_ewc.model.eval()
    del model_ewc, editor_ewc, tokens, test_batch, batch_history
    torch.cuda.empty_cache()
    
except Exception as e:
    import traceback
    print(f"✗ ft_ewc editor test failed: {e}")
    traceback.print_exc()
    torch.cuda.empty_cache()


In [ ]:
# Test Editor: ft_retrain (finetune with retraining)
try:
    torch.cuda.empty_cache()
    
    # Setup config for ft_retrain editor - uses YAML configs automatically
    config_retrain = configure_args(argparse.Namespace(
        config=cfg_path,
        editor="ft_retrain",
        inner_params=[],  # Will use YAML preset or auto-select if empty
        dataset_name="aokvqa",
        model_name=MODELNAME,  # Short name loads model/{MODELNAME}.yaml automatically
    ), config_path=cfg_path)
    config_retrain.device = config.device
    
    # Auto-select layer if not provided (from YAML or auto-select)
    if not getattr(config_retrain.model, 'inner_params', []) or len(config_retrain.model.inner_params) == 0:
        model_retrain = get_model(config_retrain)
        suggestions = explore_layers(model_retrain.model, top_k=1)
        if suggestions:
            config_retrain.model.inner_params = [suggestions[0]]
            print(f"Auto-selected layer: {config_retrain.model.inner_params[0]}")
        else:
            raise ValueError("No suitable layers found")
    else:
        model_retrain = get_model(config_retrain)
    
    # Load editor
    editor_retrain = get_editor(config_retrain, model_retrain, config_retrain.device)
    
    # COMMENTED OUT: Verify parameter exists in editor's model
    # Validation not needed - auto-selection/YAML configs provide correct layer names
    # editor_params = dict(editor_retrain.model.named_parameters())
    # if editor_retrain.pnames[0] not in editor_params:
    #     print(f"⚠ Parameter '{editor_retrain.pnames[0]}' not found in editor's model")
    #     layer_parts = editor_retrain.pnames[0].split('.')
    #     key_parts = [p for p in layer_parts if p and not p.isdigit()]
    #     matches = [n for n in editor_params.keys() 
    #                if all(part.lower() in n.lower() for part in key_parts[-3:])]
    #     if matches:
    #         corrected_name = matches[0]
    #         config_retrain.model.inner_params = [corrected_name]
    #         print(f"  Using corrected layer: {corrected_name}")
    #         editor_retrain = get_editor(config_retrain, model_retrain, config_retrain.device)
    #     else:
    #         fallback = list(editor_params.keys())[0]
    #         config_retrain.model.inner_params = [fallback]
    #         print(f"  Using fallback layer: {fallback}")
    #         editor_retrain = get_editor(config_retrain, model_retrain, config_retrain.device)
    
    print(f"✓ Editor 'ft_retrain' loaded")
    print(f"  Training layer: {editor_retrain.pnames}")
    print(f"  Retrain memory: {editor_retrain.retrain_memory}")
    
    # Prepare batch
    train_dataset.set_dataloader(
        task="mc",
        with_rationale=False,
        batch_size=1,
        shuffle=False,
    )
    test_batch = next(iter(train_dataset.loader))
    tokens = model_retrain.prepare_training_batch(test_batch)
    
    print(f"✓ prepare_training_batch() output keys: {list(tokens.keys())}")
    
    # Create batch history (retrain uses this but doesn't call retrain() automatically)
    batch_history = []
    for i, batch in enumerate(train_dataset.loader):
        if i >= 1:
            break
        tokens_hist = model_retrain.prepare_training_batch(batch)
        batch_history.append(tokens_hist)
    
    # Test editor.edit()
    model_retrain.model.train()
    
    print("\nRunning editor.edit() (retrain)...")
    editor_retrain.edit(config_retrain, tokens, batch_history)
    
    print(f"✓ editor.edit() (retrain) completed successfully!")
    print(f"  Loss history length: {len(editor_retrain.losses)}")
    if editor_retrain.losses:
        print(f"  Final loss: {editor_retrain.losses[-1]:.4f}")
    
    # Clean up
    model_retrain.model.eval()
    del model_retrain, editor_retrain, tokens, test_batch, batch_history
    torch.cuda.empty_cache()
    
except Exception as e:
    import traceback
    print(f"✗ ft_retrain editor test failed: {e}")
    traceback.print_exc()
    torch.cuda.empty_cache()


## Summary

All editors (ft, ft_ewc, ft_retrain) should now work with `prepare_training_batch()`!

**Next Steps:**
1. Explore layer names for your other models (qwen3, blip)
2. Write appropriate `inner_params` in your config files
3. Ready to create full finetuning training script!
